In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# simple model to suggest next word based on previous two words
# approach - using multilayer preceptron 

# for computational reasons I will use only the first 10, 000 sentences 

with open('data.txt' , 'r') as f:
    
    file = [line.rstrip() for line in f.readlines()]


file[:10]

['እንዴት በታዛቢ ኮሚሽነሩና በአራተኛው ረዳት ዳኛ ጎሉ ሊሻር ቻለ ህጉ ይደግፋቸዋል መልሱ አይደግፋቸውም ነው ።',
 'ከእኔ የሚፈሱ እንባዎችም ይለወጣሉ ፣ እናም የሚሰማቸው ህመም ሊቋቋሙት የማይችሉት ይ ።',
 'አሁን የሚያስፈልገን ካለፈው ።',
 'ኢትዮጵያ የዶክተር ፍቅሩ ማሩ የማረሚያ ቤት ትዝታ እና እቅዳቸው በቅርቡ በምህረት የተፈቱት የልብ ቀዶ ሕክምና ስፔሻሊስት ዶክተር ፍቅሩ ማሩ የሳቸው ሕክምና እና ክትትል ያስፈልጋቸው የነበሩ ከ 20 የሚልቁ ሕሙማን በእስር ማረሚያ ቤት በቆዩባቸው ዓመታት ውስጥ መሞታቸውን አስታወቁ ::',
 'የሴቶች መድረክ ክፍል 11 : via YouTube ።',
 'በወቅቱ ፀረ ተባይ መድሀኒት ካላስረጨህ ፣ በሚፈለገው ጊዜ በሁለት ርጭት የሚቆመው ተባይ ወቅቱ ካለፈ በኋላ የምትረጭ ከሆነ አስር ርጭት ሊፈልግ ይችላል ፡፡',
 'February 10 2020 አባይ ሚዲያ የካቲት 02 ፤2012 በትግራይ ክልል ደቡባዊ ዞን የተለያዩ አካባቢዎች የአንበጣ መንጋ ዳግም መታየቱን የትግራይ ክልል ግብርናና ገጠር ልማት ቢሮ አስታወቀ ።',
 'ደረጃ (2 3) የሚባለው የካንሰር ሴሉ በአቅራቢያው ወደ አሉ አካባቢዎች ተሰራጭቶ የሚገኝ ነው ፡፡',
 'ቤቶች ኮርፖሬሽን ለከፍተኛ የመንግሥት ባለሥልጣናት በአንድ ሥፍራ 53 ቪላዎች ሊገነባ ነው ።',
 'Ethiopian Historians AAU ጨዋታ ዶ/ር አብዱልሰመድ ፕሮፌሰርሺፕ ፈልጎ ዶ/ር ምርዕድን ሊያስፈርም ሄደ ጋሽ መርዕድ «ላንተ PHDውም ሲበዛ ነው» ሲሉት «ፋክ ዩ» ብሎ ።']

In [2]:
def contain_only_amharic_characters(sentences):
    
    for char in sentences:
        if char == ' ':
            continue 
        if not (ord('\u1200') <= ord(char) <= ord('\u137F')):
            return False
    
    return True

cleaned_dataset = [sentences for sentences in file if contain_only_amharic_characters(sentences)]
cleaned_dataset[:10]

['እንዴት በታዛቢ ኮሚሽነሩና በአራተኛው ረዳት ዳኛ ጎሉ ሊሻር ቻለ ህጉ ይደግፋቸዋል መልሱ አይደግፋቸውም ነው ።',
 'ከእኔ የሚፈሱ እንባዎችም ይለወጣሉ ፣ እናም የሚሰማቸው ህመም ሊቋቋሙት የማይችሉት ይ ።',
 'አሁን የሚያስፈልገን ካለፈው ።',
 'በወቅቱ ፀረ ተባይ መድሀኒት ካላስረጨህ ፣ በሚፈለገው ጊዜ በሁለት ርጭት የሚቆመው ተባይ ወቅቱ ካለፈ በኋላ የምትረጭ ከሆነ አስር ርጭት ሊፈልግ ይችላል ፡፡',
 'እኛም ባስመዘገብነው ውጤት ወደፊት ለመጓዝ የምንችልበትን በራስ መተማመን ጨምረናል ፡፡',
 'የችግራችን ሁሉ አስኳል የዴሞክራሲ ዕጦት ነው ብዬ አምናለሁና ።',
 'እኔስ የማውቀው ይሄን ነው ፤ እግዚኣብሄር ለእኔስ መልካም ነው ፨',
 'በቤኒሻንጉል ጉሙዝ ክልል የሚኖሩ የአማራ ተወላጆች በአካባቢው የመንግስት ታጣቂዎች ተደጋጋሚ ግፍና በደል ሲደርስባቸው ቆይቷል ።',
 'በፊት በፊት አልታዘዝም ብዬ ከደብተሬ ጋር ስጣበቅ በመጠኑም ቢሆን ቅር ይላቸው ነበር ፤ በኋላ ላይ እንዲያውም ያበረታቱኝ ጀመር ፡፡',
 'የተባበሩት መንግስታት ድርጅት የጸጥታው ጥበቃ ምክር ቤት አባላት የልኡካን ቡድን የኢትዮጵያንና የኤርትራን የድንበር አካባቢዎች ተዘዋውሮ ለመጎብኘትና ከየአገሮቹ መሪዎች ጋር የአልጀርሱ የሰላም ስምምነትተግባራዊ በሚሆንበት ሁኔታ ላይ ለመወያየት አዲስ አበባ የገባው ባለፈው አርብ ነበር ።']

In [3]:
max_size = 1000
data = cleaned_dataset[:max_size]

In [4]:
data = [line.split() for line in data]

dictionary = set(w for line in data for w in line) # 8300 words or punctuations for max_size = 1000

i = 1
wtoi = {'.':0}
for w in dictionary:
    wtoi[w] = i
    i += 1

itow = {i:w for w, i in wtoi.items()}

len(wtoi)

8301

In [32]:

def build_dataset(sentences):
    
    block_size = 2
    X, Y = [] , []
    
    for sentences in data:
    
        context = [0] * block_size

        for word in sentences + ['.']:

            ix = wtoi[word]
            X.append(context)
            Y.append(ix)

    #         print(" ".join(itow[i] for i in context), '----->' , itow[ix])

            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    
    return X, Y

import random 
random.seed(42)

random.shuffle(data)

n1 = int(0.8 * len(data))
n2 = int(0.9 * len(data))

Xtr , Ytr = build_dataset(data[:n1])
Xdev, Ydev = build_dataset(data[n1:n2])
Xtest, Ytest = build_dataset(data[n2:])


    
    

In [33]:
Xtr.shape , Xtr.dtype, Ytr.shape, Ytr.dtype

(torch.Size([16256, 2]), torch.int64, torch.Size([16256]), torch.int64)

In [55]:
words = 8301


g = torch.Generator().manual_seed(2147483647)
C = torch.randn(words, 5, generator = g).float()
W1 = torch.randn((10, 100), generator = g)
b1 = torch.randn(100, generator = g)
W2 = torch.randn((100, words), generator = g)
b2 = torch.randn(words, generator = g)

In [56]:
parameters = [C, W1, b1, W2, b2]

sum(p.nelement() for p in parameters)

881006

In [57]:
for p in parameters:
    p.requires_grad = True


In [78]:

for i in range(100):
    
    # mini batch training 
    
    ix = torch.randint(0, Xtr.shape[0], (100, ))
    
    emb = C[Xtr[ix]]

    h = torch.tanh(emb.view(-1 , 10) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
#     print(loss.item())
    for p in parameters:
        p.grad = None

    loss.backward()
    
    
    lr = 0.002
    for p in parameters:
        p.data += -lr * p.grad
        
loss.item()

4.331472396850586

In [79]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 10) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss.item()

5.070680618286133

In [101]:
# generating samples 

g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(5):
    
    out = []
    context = [0] * block_size
    
    while True:
        
        emb = C[torch.tensor([context])]
        
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        
        logits = torch.matmul(h.view(1, -1) ,  W2.view(-1, 8301)) + b2
        
        probs = F.softmax(logits, dim = 1)
        ix = torch.multinomial(probs, num_samples=1, generator = g).item()
        
        context = context[1:] + [ix]
        out.append(ix)
        
        if ix == 0:
            break
        
    print(" ".join(itow[i] for i in out))

ተገልጿል ፡፡ .
እንደሚያሳውቁ አሶሽየትድ ነን የሚገኙት በፊትም ማህበራት ነው ይላሉ ። .
ሱስንዮስን ። .
ሊመጣ አዛዥ ፤ የማህበራዊ ዳር ይህ የአየር ችግር ሰው ካዳነች ነው ፡፡ .
ምን ዓረብ አገሪቱ የመከተል ጲላጦስ ። .
